In [3]:
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ScanConfig:
    root: Path = Path(".")
    skip_dirs: frozenset[str] = frozenset({".git", "venv", ".venv", "node_modules"})
    trash_dirnames: frozenset[str] = frozenset(
        {
            "__pycache__",
            ".pytest_cache",
            ".mypy_cache",
            ".ruff_cache",
            ".ipynb_checkpoints",
            ".cache",
        }
    )
    trash_filenames: frozenset[str] = frozenset({".DS_Store", "Thumbs.db"})
    trash_suffixes: frozenset[str] = frozenset({".pyc", ".pyo", ".log"})


def _dir_size_bytes(path: Path) -> int:
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            try:
                total += p.stat().st_size
            except OSError:
                pass
    return total


def find_cache_only_dirs(cfg: ScanConfig = ScanConfig()):
    """Return directories whose contents (recursively) are only cache/trash artifacts.

    A directory qualifies if it has at least one child entry and all entries under it are:
    - trash directories (e.g. __pycache__, .pytest_cache, ...)
    - trash files (e.g. .pyc, .DS_Store, ...)
    """

    candidates: list[Path] = []

    for dirpath, dirnames, filenames in os.walk(cfg.root):
        path = Path(dirpath)

        # Prune skipped trees early
        dirnames[:] = [d for d in dirnames if d not in cfg.skip_dirs]
        if any(part in cfg.skip_dirs for part in path.parts):
            dirnames[:] = []
            continue

        # Decide whether THIS directory contains anything beyond trash directly inside it.
        nontrash_direct_children = []
        for d in dirnames:
            if d not in cfg.trash_dirnames:
                nontrash_direct_children.append(("dir", d))
        for f in filenames:
            if f in cfg.trash_filenames:
                continue
            if Path(f).suffix in cfg.trash_suffixes:
                continue
            nontrash_direct_children.append(("file", f))

        if (dirnames or filenames) and not nontrash_direct_children:
            candidates.append(path)

    # Keep only the highest-level directories (if a parent qualifies, drop its children)
    candidates = sorted(candidates, key=lambda p: (len(p.parts), p.as_posix()))
    kept: list[Path] = []
    for p in candidates:
        if any(p == k or p.is_relative_to(k) for k in kept):
            continue
        kept.append(p)

    return kept


cfg = ScanConfig()
dirs = find_cache_only_dirs(cfg)
print(f"Found {len(dirs)} cache-only directories (excluding {sorted(cfg.skip_dirs)}):")
for d in dirs:
    rel = d.resolve().relative_to(cfg.root.resolve())
    size_kb = _dir_size_bytes(d) / 1024
    print(f"- {rel.as_posix()}  (~{size_kb:.1f} KiB)")


Found 2 cache-only directories (excluding ['.git', '.venv', 'node_modules', 'venv']):
- __pycache__  (~156.6 KiB)
- docs/__pycache__  (~6.6 KiB)
